In [39]:
import os
import sys
import torch
import torchvision
import yaml
import tqdm

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import torchvision.models as models

from PIL import Image
from omegaconf import OmegaConf
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix

from utils.data_utils import get_data, get_gen

# set directory to parent directory
os.chdir("/Users/jorgegoncalves/Desktop/Repositories/Master_Thesis/treevae")

# print current working directory
print("Current Working Directory:", os.getcwd())


Current Working Directory: /Users/jorgegoncalves/Desktop/Repositories/Master_Thesis/treevae


# Pretrain CIFAR-10 Classification Model

In [40]:
configs = {
    "data": {
        "data_name": "cifar10",
        "num_clusters_data": 10,
    },
    "training": {
        "batch_size": 256,
        "augment": False,
        "augmentation_method": 'simple',
        "aug_decisions_weight": 1,
    },
    "globals": {
        "seed": 42,
    }
}

In [41]:

# Dataset
trainset, trainset_eval, testset = get_data(configs)

gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)

gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)

y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


In [42]:
print("\nTrainset shape:", trainset.dataset.data.shape)
print("Number of samples in trainset:", len(gen_train.dataset))

print("\nTrainset eval shape:", trainset_eval.dataset.data.shape)
print("Number of samples in trainset eval:", len(gen_train_eval.dataset))

print("\nTestset shape:", testset.dataset.data.shape)
print("Number of samples in testset:", len(gen_test.dataset))


Trainset shape: (50000, 32, 32, 3)
Number of samples in trainset: 50000

Trainset eval shape: (50000, 32, 32, 3)
Number of samples in trainset eval: 50000

Testset shape: (10000, 32, 32, 3)
Number of samples in testset: 10000


In [45]:
# device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model 
model = models.resnet50(pretrained=True)
model = model.to(device)

# Loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Epochs
n_epochs = 10

# Training
model.train()
for epoch in tqdm.tqdm(range(n_epochs)):
    for i, (x_batch, y_batch) in enumerate(gen_train):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()

        y_pred = model(x_batch)

        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()



  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 0, Batch: 0, Loss: 13.30945110321045, Loss eval: 14.781968116760254
Epoch: 0, Batch: 10, Loss: 1.7483398914337158, Loss eval: 6.821011066436768
Epoch: 0, Batch: 20, Loss: 1.4735313653945923, Loss eval: 3.68410587310791
Epoch: 0, Batch: 30, Loss: 1.1803196668624878, Loss eval: 2.0867908000946045
Epoch: 0, Batch: 40, Loss: 1.126456618309021, Loss eval: 1.165757417678833
Epoch: 0, Batch: 50, Loss: 1.034407615661621, Loss eval: 0.9295726418495178
Epoch: 0, Batch: 60, Loss: 0.8127579689025879, Loss eval: 0.9369191527366638
Epoch: 0, Batch: 70, Loss: 0.7655667066574097, Loss eval: 0.7090275883674622
Epoch: 0, Batch: 80, Loss: 0.8234173059463501, Loss eval: 0.7539400458335876
Epoch: 0, Batch: 90, Loss: 0.6559050679206848, Loss eval: 0.6287984848022461
Epoch: 0, Batch: 100, Loss: 0.683707594871521, Loss eval: 0.7142171859741211
Epoch: 0, Batch: 110, Loss: 0.6627610325813293, Loss eval: 0.5552857518196106
Epoch: 0, Batch: 120, Loss: 0.7752763032913208, Loss eval: 0.6570124626159668
Epoch

 10%|█         | 1/10 [08:02<1:12:23, 482.64s/it]

Epoch: 1, Batch: 0, Loss: 0.5381361842155457, Loss eval: 0.6641537547111511
Epoch: 1, Batch: 10, Loss: 0.43552088737487793, Loss eval: 0.6350979208946228
Epoch: 1, Batch: 20, Loss: 0.5968358516693115, Loss eval: 0.5085350871086121
Epoch: 1, Batch: 30, Loss: 0.5691208243370056, Loss eval: 0.314471036195755
Epoch: 1, Batch: 40, Loss: 0.5933914184570312, Loss eval: 0.24661381542682648
Epoch: 1, Batch: 50, Loss: 0.5082234144210815, Loss eval: 0.2565740644931793
Epoch: 1, Batch: 60, Loss: 0.4509425163269043, Loss eval: 0.36089178919792175
Epoch: 1, Batch: 70, Loss: 0.40546321868896484, Loss eval: 0.32390159368515015
Epoch: 1, Batch: 80, Loss: 0.4420905113220215, Loss eval: 0.3561154305934906
Epoch: 1, Batch: 90, Loss: 0.4118736982345581, Loss eval: 0.34248292446136475
Epoch: 1, Batch: 100, Loss: 0.442757785320282, Loss eval: 0.44268253445625305
Epoch: 1, Batch: 110, Loss: 0.41175124049186707, Loss eval: 0.4644497036933899
Epoch: 1, Batch: 120, Loss: 0.561725378036499, Loss eval: 0.343953430

 20%|██        | 2/10 [15:43<1:02:38, 469.87s/it]

...

Epoch: 8, Batch: 0, Loss: 0.09610973298549652, Loss eval: 0.1789723038673401
Epoch: 8, Batch: 10, Loss: 0.08798546344041824, Loss eval: 0.1118566244840622
Epoch: 8, Batch: 20, Loss: 0.12171107530593872, Loss eval: 0.22406600415706635
Epoch: 8, Batch: 30, Loss: 0.04698598384857178, Loss eval: 0.20194582641124725
Epoch: 8, Batch: 40, Loss: 0.08178284019231796, Loss eval: 0.14342159032821655
Epoch: 8, Batch: 50, Loss: 0.07755342125892639, Loss eval: 0.1493884176015854
Epoch: 8, Batch: 60, Loss: 0.08586625754833221, Loss eval: 0.22761036455631256
Epoch: 8, Batch: 70, Loss: 0.06592929363250732, Loss eval: 0.15470370650291443
Epoch: 8, Batch: 80, Loss: 0.1432628631591797, Loss eval: 0.2500542998313904
Epoch: 8, Batch: 90, Loss: 0.10068640112876892, Loss eval: 0.19756579399108887
Epoch: 8, Batch: 100, Loss: 0.08520018309354782, Loss eval: 0.1274845153093338
Epoch: 8, Batch: 110, Loss: 0.15604794025421143, Loss eval: 0.20999689400196075
Epoch: 8, Batch: 120, Loss: 0.18599183857440948, Loss eva

 90%|█████████ | 9/10 [1:11:10<07:29, 449.48s/it]

Epoch: 9, Batch: 0, Loss: 0.07395273447036743, Loss eval: 0.19991903007030487
Epoch: 9, Batch: 10, Loss: 0.05668143555521965, Loss eval: 0.06436412781476974
Epoch: 9, Batch: 20, Loss: 0.11903352290391922, Loss eval: 0.10313810408115387
Epoch: 9, Batch: 30, Loss: 0.0642690360546112, Loss eval: 0.180341437458992
Epoch: 9, Batch: 40, Loss: 0.08368503302335739, Loss eval: 0.17099255323410034
Epoch: 9, Batch: 50, Loss: 0.07454068213701248, Loss eval: 0.2434377521276474
Epoch: 9, Batch: 60, Loss: 0.10946515202522278, Loss eval: 0.20723840594291687
Epoch: 9, Batch: 70, Loss: 0.06503120809793472, Loss eval: 0.24209807813167572
Epoch: 9, Batch: 80, Loss: 0.10434208065271378, Loss eval: 0.12630189955234528
Epoch: 9, Batch: 90, Loss: 0.07122856378555298, Loss eval: 0.2048916518688202
Epoch: 9, Batch: 100, Loss: 0.08122169971466064, Loss eval: 0.18716742098331451
Epoch: 9, Batch: 110, Loss: 0.09279803931713104, Loss eval: 0.21297121047973633
Epoch: 9, Batch: 120, Loss: 0.1604350209236145, Loss eva

 90%|█████████ | 9/10 [1:16:24<08:29, 509.38s/it]


StopIteration: 

In [46]:
# Model Evaluation
model.eval()

with torch.no_grad():
    for i, (x, y) in enumerate(gen_test):
        if i == 0:
            y_pred = model(x)
        else:
            y_pred = torch.cat([y_pred, model(x)], dim=0)

y_pred = y_pred.argmax(dim=1).numpy()

In [48]:

print("y_test shape:", y_test.shape)

# accuracy
acc = np.mean(y_pred == y_test)
print("Accuracy:", acc)

# confusion matrix


y_test shape: (10000,)
Accuracy: 0.7982


In [49]:
torch.save(model.state_dict(), "models/resnet50_cifar10.pt")

In [52]:
model2 = torch.load("models/resnet50_cifar10.pt")
model2.eval()

with torch.no_grad():
    for i, (x, y) in enumerate(gen_test):
        if i == 0:
            y_pred = model2(x)
        else:
            y_pred = torch.cat([y_pred, model2(x)], dim=0)

TypeError: 'collections.OrderedDict' object is not callable

In [56]:
# load resnet50 model with weights from classifier_pretraining/resnet50_cifar10.pth

model2 = models.resnet50(pretrained=False)
model2.load_state_dict(torch.load("classifier_pretraining/resnet50_cifar10.pth"))
model2.eval()

with torch.no_grad():
    for i, (x, y) in enumerate(gen_test):
        if i == 0:
            y_pred = model2(x)
        else:
            y_pred = torch.cat([y_pred, model2(x)], dim=0)

y_pred = y_pred.argmax(dim=1).numpy()

# accuracy
acc = np.mean(y_pred == y_test)
print("Accuracy:", acc)


/opt/anaconda3/envs/treevae2/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/treevae2/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Accuracy: 0.7838


# Pretrain CIFAR-10 MNIST Model

In [57]:
dataset = 'mnist'


# Configurations
configs = {
    "data": {
        "data_name": dataset,
        "num_clusters_data": 10,
    },
    "training": {
        "batch_size": 256,
        "augment": False,
        "augmentation_method": 'simple',
        "aug_decisions_weight": 1,
    },
    "globals": {
        "seed": 42,
    }
}

In [59]:
# Dataset
trainset, trainset_eval, testset = get_data(configs)

gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)

gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)

y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

In [61]:
print("\nTrainset shape:", trainset.dataset.data.shape)


Trainset shape: torch.Size([60000, 28, 28])


In [64]:
# device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
model = models.resnet50(pretrained=True)

# change model to fit mnist
model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
model = model.to(device)

# Loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Epochs
n_epochs = 10

# Training
model.train()
for epoch in tqdm.tqdm(range(n_epochs)):
    for i, (x_batch, y_batch) in enumerate(gen_train):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()

        y_pred = model(x_batch)

        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()



/opt/anaconda3/envs/treevae2/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/treevae2/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
 50%|█████     | 5/10 [42:45<42:45, 513.02s/it]  


KeyboardInterrupt: 

# Pretrain CIFAR-10 FashionMNIST Model